# CGSmiles polymer authoring

Edit the three tagged cells only: graph, chemistry/maps/templates, and output settings plus build. This offline helper writes charged SDFs and a provided_molecules YAML snippet; runtime simulation builds do not resolve CGSmiles. Each coarse edge must resolve to one atomistic interfragment bond, with edge order interpreted as bond order (1.0, 1.5 aromatic, 2.0, or 3.0).

In [ ]:
from pathlib import Path
import mbuild as mb, networkx as nx
from polyzymd.builders.polymer_authoring import build_and_export_polymer
def project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / 'config.yaml').is_file():
            return candidate
    raise RuntimeError('Run this notebook from a PolyzyMD project root or its notebooks/ directory')
project_root = project_root()


In [ ]:
# User choice: specify the coarse residue graph
cg_graph = nx.Graph()
cg_graph.add_node(0, fragname='CORE', position=(0.0, 0.0, 0.0))
cg_graph.add_node(1, fragname='ARM', position=(0.55, 0.0, 0.0))
cg_graph.add_node(2, fragname='ARM', position=(-0.28, 0.48, 0.0))
cg_graph.add_node(3, fragname='ARM', position=(-0.28, -0.48, 0.0))
cg_graph.add_edges_from([(0, 1, {'order': 1}), (0, 2, {'order': 1}), (0, 3, {'order': 1})])


In [ ]:
# User choice: CGSmiles fragments plus explicit atom, port, and leaving maps
def methyl_template(name, vectors):
    t = mb.Compound(name=name); c = mb.Compound(name='C', element='C', pos=[0, 0, 0]); t.add(c, label='C')
    for i, pos in enumerate(([0.11, 0, 0], [-0.04, 0.10, 0], [-0.04, -0.05, 0.09], [-0.04, -0.05, -0.09]), 1):
        h = mb.Compound(name=f'H{i}', element='H', pos=pos); t.add(h, label=f'H{i}'); t.add_bond((c, h), bond_order=1)
    for i, vec in enumerate(vectors, 1):
        t.add(mb.Port(anchor=c, orientation=vec, separation=0.08), label=f'port_{i}')
    return t
fragments = '{#CORE=[$br]C([$br])[$br],#ARM=[$br]C}'
aa_templates = {'CORE': methyl_template('CORE', [(1, 0, 0), (-0.5, 0.86, 0), (-0.5, -0.86, 0)]), 'ARM': methyl_template('ARM', [(-1, 0, 0)])}
atom_maps = {'CORE': {0: 'C'}, 'ARM': {0: 'C'}}
port_maps = {'CORE': {'$br1': ['port_1', 'port_2', 'port_3']}, 'ARM': {'$br1': 'port_1'}}
leaving_maps = {'CORE': {'$br1': ['H1', 'H2', 'H3']}, 'ARM': {'$br1': 'H1'}}


In [ ]:
# User choice: output settings, then build/export
output_name = 'branched_polymer'
charge_method = 'gasteiger'
clash_threshold_nm = 0.10
output_sdf = project_root / 'generated_molecules' / f'{output_name}.sdf'
result = build_and_export_polymer(cg_graph=cg_graph, fragments=fragments, aa_templates=aa_templates, atom_maps=atom_maps, port_maps=port_maps, leaving_maps=leaving_maps, output_sdf=output_sdf, charge_method=charge_method, name=output_name, clash_threshold_nm=clash_threshold_nm, yaml_base_path=project_root)
print(f"Wrote charged SDF: {result['sdf_path']}")
print(f"Wrote YAML snippet: {result['yaml_path']}")
print('Paste this full polymers block for provided-only mode. If you already use dynamic or fragments mode, merge only the provided_molecules list into that existing polymers block.')
print(result['yaml_snippet'])
